# **13 - Renewal-only study (separate product)**

**Question.** *What will a tenant pay on a future renewal, given the current price?*

**Framing.**
- This notebook is a **separate, clearly labelled product**. It uses `previous_deposit_10k_krw` (Jeonse) and `previous_monthly_rent_10k_krw` (Wolse), which rule 5 of the modeling protocol forbids for the main models. It is an explicit, isolated exception documented in `CLAUDE.md`.
- The previous price is **not** added to the main models, and the runs saved here carry `scope = "renewal_only"` in their meta and are skipped by the main leaderboard.
- It **does not apply to new contracts**, which have no previous price, and it is **not part of the market projection**.
- That the previous price is available at prediction time is a **domain assumption**: for a future renewal the parties know the current price.

**Plan.**
1. Audit of the `previous_*` columns and a documented plausibility rule (rows are flagged, not removed from the main dataset).
2. Policy analysis of the new / previous ratio by `renewal_right_used`.
3. Three models on renewal rows, both tracks: (a) carry-forward (new = previous), (b) previous x median ratio of the group (renewal right x building type, fit window only), (c) HistGradientBoosting with median loss on `log(new / previous)` with the allowed features plus `log(previous)`. They are evaluated on the two pseudo-tests (fit before 2023 and test 2023; fit before 2024 and test 2024) and on 2025 once, and compared with the main winner on the same rows.

**Evaluation sample and plausibility rule.** The rule has three parts: a positive previous price, a previous price not above a hard cap of the track, and a new / previous ratio inside a domain-based multiple. The first two use information known at prediction time and define the **evaluation rows**; the ratio part uses the new price, so it is applied to **training rows only** (to clean the fit) and never to select evaluation rows. The notebook also reports the WAPE on evaluation rows that pass the ratio part as a sensitivity check.

2025 was inspected in earlier iterations of this project (see the README).

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'rent_model').exists())
sys.path.insert(0, str(ROOT / 'src'))

import json

import numpy as np
import pandas as pd

from rent_model import config
from rent_model.boosting import tune_extended
from rent_model.data import load_clean
from rent_model.features import build_features, select_track
from rent_model.io import save_run
from rent_model.metrics import all_metrics, wape
from rent_model.pseudotest import WINDOWS, check_windows, window_split
from rent_model.recipes import ModelRecipe
from rent_model.renewal import (CAP_TOLERANCE, CarryForward, GroupRatioMedian, LogRatioBoost, PRICE_CAP, RATIO_BOUNDS, RENEWAL_CAP, SCOPE,
                                SEARCH_SPACE, PREVIOUS_COLUMN, flag_previous, previous_price, ratio_wape_scorer, renewal_features)
from rent_model.boosting import required_columns
from rent_model.splits import temporal_split, window_folds

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
N_ITER = 10       # candidates sampled by each random search (3 folds each)
print('Random seed:', config.RANDOM_SEED, '| scope of this study:', SCOPE, '| hard caps (10,000 KRW):', PRICE_CAP, '| ratio bounds:', RATIO_BOUNDS)

Random seed: 42 | scope of this study: renewal_only | hard caps (10,000 KRW): {'jeonse': 1000000.0, 'wolse': 5000.0} | ratio bounds: (0.2, 5.0)


## Data

In [2]:
df = load_clean()
check_windows()
FIT_END = {w['name']: w['fit_end'] for w in WINDOWS}
FIT_END['final'] = config.TEST_START
data = {}
for track, spec in config.TRACKS.items():
    d = select_track(df, track)
    train, test = temporal_split(d)
    data[track] = dict(df=d, train=train, test=test, target=spec['target'], flags=flag_previous(d, track))
    print(f"{track}: {len(d):,} rows (TRAIN {len(train):,}, TEST 2025 {len(test):,}); target {spec['target']}; previous price column {PREVIOUS_COLUMN[track]}")

jeonse: 176,783 rows (TRAIN 138,848, TEST 2025 37,935); target deposit_10k_krw; previous price column previous_deposit_10k_krw
wolse: 210,543 rows (TRAIN 152,714, TEST 2025 57,829); target monthly_rent_10k_krw; previous price column previous_monthly_rent_10k_krw


## 1) Audit of the previous_* columns and plausibility rule

In [3]:
rows = []
for track in config.TRACKS:
    d, f, test = data[track]['df'], data[track]['flags'], data[track]['test']
    prev, ren = previous_price(d, track), f['is_renewal']
    row = {'track': track, 'track rows': len(d), 'renewal rows': int(ren.sum()), 'renewal share of the track (%)': ren.mean() * 100,
           'renewal share of TEST 2025 (%)': f.loc[test.index, 'is_renewal'].mean() * 100,
           'non-renewal rows with a previous price > 0': int((~ren & (prev > 0)).sum()),
           'renewals: previous missing or <= 0': int((ren & f['previous_missing']).sum()), 'renewals: previous above the hard cap': int((ren & f['above_cap']).sum()),
           'renewals: ratio outside the bounds': int((ren & f['ratio_outside']).sum()), 'evaluation rows (input rule)': int(f['input_ok'].sum()),
           'training rows (all rules)': int(f['fully_ok'].sum()), 'renewals flagged by any rule': int((ren & ~f['fully_ok']).sum())}
    rows.append(row)
audit = pd.DataFrame(rows).set_index('track')
display(audit.T)
for track in config.TRACKS:
    d, f = data[track]['df'], data[track]['flags']
    prev, ren = previous_price(d, track), f['is_renewal']
    print(f"{track}: largest previous price among renewals {prev[ren].max():,.0f}; largest current price of the track {d[data[track]['target']].max():,.0f}; hard cap {PRICE_CAP[track]:,.0f}")
w = data['wolse']
conv = w['flags']['is_renewal'] & w['flags']['previous_missing'] & (w['df']['previous_deposit_10k_krw'] > 0)
print(f"wolse renewals without a previous monthly rent but with a previous deposit (previous contract without monthly rent): {int(conv.sum()):,}")
print('Nothing is removed from the main dataset: the rows are flagged for this study only.')

track,jeonse,wolse
track rows,"176,783.0000","210,543.0000"
renewal rows,"56,989.0000","35,629.0000"
renewal share of the track (%),32.2367,16.9224
renewal share of TEST 2025 (%),39.8128,21.1347
non-renewal rows with a previous price > 0,0.0000,4.0000
renewals: previous missing or <= 0,3.0000,"5,195.0000"
renewals: previous above the hard cap,413.0000,230.0000
renewals: ratio outside the bounds,566.0000,387.0000
evaluation rows (input rule),"56,573.0000","30,204.0000"
training rows (all rules),"56,420.0000","30,047.0000"


jeonse: largest previous price among renewals 2,200,000,000; largest current price of the track 750,000; hard cap 1,000,000
wolse: largest previous price among renewals 4,280,000; largest current price of the track 4,000; hard cap 5,000
wolse renewals without a previous monthly rent but with a previous deposit (previous contract without monthly rent): 5,194
Nothing is removed from the main dataset: the rows are flagged for this study only.


**Interpretation.** Renewals are 56,989 rows in Jeonse (32.2367% of the track and 39.8128% of the TEST rows of 2025) and 35,629 rows in Wolse (16.9224% and 21.1347%). The previous price exists only for renewals, with 4 exceptions in Wolse (4 non-renewal rows with a previous price) and none in Jeonse. In Jeonse almost every renewal has a usable previous deposit: 3 are missing or not positive, 413 are above the hard cap of 1,000,000 (the largest previous deposit is 2,200,000,000 against a largest current deposit of 750,000) and 566 have a ratio outside the bounds; 569 renewals are flagged by any rule. In Wolse 5,195 renewals have no previous monthly rent (5,194 of them have a previous deposit: the previous contract had no monthly rent, so the previous rent cannot be carried forward and the track has a different product for those rows); 230 are above the hard cap of 5,000 (largest previous rent 4,280,000 against a largest current rent of 4,000) and 387 have a ratio outside the bounds; 5,582 renewals are flagged. The evaluation rows (input rule: positive previous price not above the cap) are 56,573 in Jeonse and 30,204 in Wolse, and the training rows (all rules) 56,420 and 30,047. No row is removed from the main dataset; the flags apply to this study only, and the ratio part of the rule never selects evaluation rows.

## 2) Was the previous price known before the contract date?

In [4]:
rows = []
for track in config.TRACKS:
    d, f = data[track]['df'], data[track]['flags']
    ren = d[f['is_renewal']]
    period = ren['contract_period'].astype(str).str.extract(r'^(\d\d)\.(\d\d)~')
    start = pd.to_datetime('20' + period[0] + '-' + period[1] + '-01', errors='coerce')
    gap = ((ren['contract_month'].dt.year - start.dt.year) * 12 + (ren['contract_month'].dt.month - start.dt.month)).dropna()
    rows.append({'track': track, 'renewal rows': len(ren), 'period start parsed': len(gap), 'contract month before the lease start': int((gap < 0).sum()),
                 'contract month = lease start month': int((gap == 0).sum()), 'contract month after the lease start': int((gap > 0).sum()),
                 'median months between contract month and lease start': float(gap.median())})
display(pd.DataFrame(rows).set_index('track').T)
print('The previous_* columns are filled for renewals only (see the audit); the data has no date for the previous contract.')

track,jeonse,wolse
renewal rows,"56,989.0000","35,629.0000"
period start parsed,"55,642.0000","34,635.0000"
contract month before the lease start,"33,396.0000","21,109.0000"
contract month = lease start month,"21,812.0000","12,901.0000"
contract month after the lease start,434.0000,625.0000
median months between contract month and lease start,-1.0000,-1.0000


The previous_* columns are filled for renewals only (see the audit); the data has no date for the previous contract.


**Interpretation.** The data has no date for the previous contract, so the check is indirect. For 55,642 Jeonse and 34,635 Wolse renewals the lease start can be parsed from `contract_period`: the contract month is before the lease start for 33,396 (Jeonse) and 21,109 (Wolse) renewals, equal to the start month for 21,812 and 12,901, and after the start for only 434 and 625; the median gap is -1 month in both tracks. The renewal is therefore normally registered no later than the start of the new lease, at which time the price of the contract being renewed is known to the parties, and the previous price is a property of the contract that came before it. The check cannot prove the order of the dates for the few renewals registered after the lease start, and the availability of the previous price at prediction time remains a domain assumption.

## 3) Policy analysis: ratio new / previous by renewal right

In [5]:
rows, by_year = [], []
for track in config.TRACKS:
    d, f = data[track]['df'], data[track]['flags']
    sub = d[f['fully_ok']]
    ratio = sub[data[track]['target']] / previous_price(sub, track)
    for right in ['Yes', 'No/Not reported']:
        r = ratio[sub['renewal_right_used'] == right]
        rows.append({'track': track, 'renewal right used': right, 'rows': len(r), 'median ratio': r.median(), 'p05': r.quantile(0.05), 'p95': r.quantile(0.95),
                     f'share <= {RENEWAL_CAP:.2f}': (r <= RENEWAL_CAP + 1e-9).mean(), f'share <= {RENEWAL_CAP + CAP_TOLERANCE:.3f}': (r <= RENEWAL_CAP + CAP_TOLERANCE + 1e-9).mean(),
                     'share == 1': (r == 1).mean(), f'share > {RENEWAL_CAP + CAP_TOLERANCE:.3f}': (r > RENEWAL_CAP + CAP_TOLERANCE + 1e-9).mean()})
        yy = r.groupby(sub.loc[r.index, 'contract_date'].dt.year).apply(lambda x: (x <= RENEWAL_CAP + CAP_TOLERANCE + 1e-9).mean())
        by_year.append(pd.Series(yy, name=(track, right)))
display(pd.DataFrame(rows).set_index(['track', 'renewal right used']))
print(f'share of ratios <= {RENEWAL_CAP + CAP_TOLERANCE:.3f} by contract year:')
display(pd.DataFrame(by_year))

rows  median ratio    p05    p95  share <= 1.05  share <= 1.055  share == 1  share > 1.055
track  renewal right used                                                                                             
jeonse Yes                 24938        1.0488 0.8454 1.0500         0.9859          0.9910      0.1397         0.0090
       No/Not reported     31482        1.0496 0.8696 1.3269         0.6245          0.6467      0.1901         0.3533
wolse  Yes                  9281        1.0533 0.8889 1.3600         0.4608          0.5313      0.1833         0.4687
       No/Not reported     20766        1.0533 0.6667 1.7500         0.4799          0.5159      0.2089         0.4841

share of ratios <= 1.055 by contract year:


contract_date            2022   2023   2024   2025
jeonse Yes             0.9985 0.9539 0.9965 0.9997
       No/Not reported 0.4845 0.7039 0.6940 0.6702
wolse  Yes             0.5368 0.7231 0.5233 0.4405
       No/Not reported 0.4868 0.4819 0.5384 0.5353

**Interpretation.** In Jeonse, renewals with the renewal right used have a median ratio of 1.0488 and 0.9859 of them are at or below 1.05 (0.9910 at or below 1.055, 0.0090 above); the 95th percentile of the ratio is 1.0500. Without the right (or not reported) the median is almost the same (1.0496) but only 0.6245 are at or below 1.05 and 0.3533 are above 1.055. This is consistent with a 5% limit on the increase that applies when the renewal right is used, and by year the share at or below 1.055 for the right used is 0.9985, 0.9539, 0.9965 and 0.9997 in 2022 to 2025. In Wolse the pattern is absent: the median ratio is 1.0533 in both groups and the share at or below 1.055 is 0.5313 (right used) and 0.5159, with 0.4687 and 0.4841 above, so the data do not show a 5% cap on the monthly rent; the numbers do not say why. Only what the printed shares show is stated here; the ratios of rows with the renewal right not reported may include renewals where the right was used but not recorded.

## 4) Tuning of the gradient boosting model inside each fit window

In [6]:
tuned, rows = {}, []
for track in config.TRACKS:
    d = data[track]['df']
    for name, fit_end in FIT_END.items():
        fit = d[d['contract_date'] < fit_end]
        assert fit['contract_date'].max() < fit_end
        fit = fit[flag_previous(fit, track)['fully_ok']]
        X = renewal_features(fit, track)
        y = np.log(fit[data[track]['target']].astype(float) / previous_price(fit, track))
        folds = window_folds(fit)
        t0 = time.time()
        best, table, n = tune_extended('static', X[required_columns('static') + ['log_prev']], y, folds, n_iter=N_ITER, quantile=0.5,
                                       scorer=ratio_wape_scorer, distributions=SEARCH_SPACE, extra_numeric=('log_prev',))
        tuned[(track, name)] = best
        rows.append({'track': track, 'fit window': name, 'fit rows (training rule)': n, 'folds (train/valid rows)': ' '.join(f'{len(a)}/{len(b)}' for a, b in folds),
                     'best CV WAPE': table.loc[0, 'cv_wape_mean'], 'seconds': time.time() - t0, **best})
display(pd.DataFrame(rows).set_index(['track', 'fit window']))
print('candidates per search:', N_ITER, '| grid:', SEARCH_SPACE)

fit rows (training rule)          folds (train/valid rows)  best CV WAPE  seconds  learning_rate  max_iter  max_leaf_nodes  min_samples_leaf  l2_regularization
track  fit window                                                                                                                                                                 
jeonse 2023                           16474   8820/3057 11877/2469 14346/2128        5.2868  30.9762         0.1000       200              31               100            10.0000
       2024                           28969  16474/4612 21086/4001 25087/3882       12.1506  37.2614         0.1000       200              63               100             1.0000
       final                          41350  16474/8613 25087/8031 33118/8232        9.8496  45.5912         0.1000       200              63               100             1.0000
wolse  2023                            6114       3333/1015 4348/807 5155/959       11.3573  26.5851         0.0500       200              31               100            10.0000
       2024                           12294    6114/2181 8295/2039 10334/1960       14.7135  44.6513         0.0500       400              15                20             1.0000
       final                          19804   6114/4220 10334/4129 14463/5341       12.8036  43.6307         0.0500       400              15                20             1.0000

candidates per search: 10 | grid: {'learning_rate': [0.05, 0.1], 'max_iter': [200, 400], 'max_leaf_nodes': [15, 31, 63], 'min_samples_leaf': [20, 50, 100], 'l2_regularization': [0.0, 1.0, 10.0]}


**Interpretation.** The gradient boosting model is tuned inside each fit window on the training rows of the renewal rows (16,474, 28,969 and 41,350 rows in Jeonse and 6,114, 12,294 and 19,804 in Wolse for the three windows), by random search with 10 candidates and 3 expanding-window folds, scored with the WAPE of the rebuilt new price. For the final window the fit rows are 41,350 (Jeonse) and 19,804 (Wolse) and the best CV WAPE is 9.8496% and 12.8036%. The selected settings are small trees (`max_leaf_nodes` 63 in Jeonse, 15 in Wolse for the final window) and differ between windows, which is expected with few rows.

## 5) Pseudo-tests: carry-forward, group median ratio, gradient boosting and the main winner on the same rows

In [7]:
PARAMS_MAIN = dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=255, min_samples_leaf=20, l2_regularization=0.0, min_frequency=30)   # settings of the main winners (notebook 11)
chosen = {t: json.loads((config.METRICS_DIR / f'pseudotest_v2_{t}.json').read_text(encoding='utf-8'))['selection']['extrapolating'] for t in config.TRACKS}
for t in config.TRACKS:
    assert json.loads((config.METRICS_DIR / f'{t}__{chosen[t]}.json').read_text(encoding='utf-8'))[-1]['meta']['params'] == PARAMS_MAIN
print('main winners (the same models as in the leaderboard):', chosen)

def study_models(track, name):
    return {'carry-forward': CarryForward(), 'group median ratio': GroupRatioMedian(), 'gradient boosting (log ratio, median loss)': LogRatioBoost(tuned[(track, name)])}

def evaluate_window(track, name, test_frame, main_pred):
    """Predictions of every model on the evaluation rows (input rule) of one window."""
    d = data[track]['df']
    fit = d[d['contract_date'] < FIT_END[name]]
    f_test = flag_previous(test_frame, track)
    mask = f_test['input_ok'].to_numpy()
    ev = test_frame[mask]
    out = {'y': ev[data[track]['target']].to_numpy(), 'ratio_ok': f_test.loc[ev.index, 'fully_ok'].to_numpy(), 'right': ev['renewal_right_used'].to_numpy(),
           'preds': {}}
    out['n_fit'] = {}
    for label, model in study_models(track, name).items():
        out['preds'][label] = model.fit(fit, track).predict(ev)
        out['n_fit'][label] = getattr(model, 'n_fit_', None)
    out['preds']['main winner (' + chosen[track] + ')'] = main_pred[mask]
    return out

pseudo = {}
for track in config.TRACKS:
    d = data[track]['df']
    for w in WINDOWS:
        t0 = time.time()
        fit, test_frame = window_split(d, w)
        main = ModelRecipe(chosen[track], track, params=PARAMS_MAIN).fit(fit, w['fit_end'])
        main_pred = main.predict(build_features(test_frame, track, include_time=True, include_legal_dong=True), test_frame['contract_month'])
        pseudo[(track, w['name'])] = evaluate_window(track, w['name'], test_frame, np.asarray(main_pred))
        print(f"{track} / pseudo-test {w['name']}: {len(pseudo[(track, w['name'])]['y']):,} renewal evaluation rows, done in {time.time() - t0:,.0f} s")

def metric_rows(results, label):
    rows = []
    for (track, name), r in results.items():
        for model, pred in r['preds'].items():
            m = all_metrics(r['y'], pred)
            ok = r['ratio_ok']
            rows.append({'track': track, 'evaluation': name, 'model': model, 'rows': len(r['y']), 'WAPE %': m['wape_pct'], 'MdAPE %': m['mdape_pct'],
                         'aggregate bias %': m['aggregate_bias_pct'], 'WAPE % (ratio rule also passed)': wape(r['y'][ok], pred[ok])})
    return pd.DataFrame(rows).set_index(['track', 'evaluation', 'model'])

pooled = {}
for track in config.TRACKS:
    parts = [pseudo[(track, w['name'])] for w in WINDOWS]
    pooled[(track, 'pseudo-tests 2023+2024')] = {'y': np.concatenate([p['y'] for p in parts]), 'ratio_ok': np.concatenate([p['ratio_ok'] for p in parts]),
                                                'right': np.concatenate([p['right'] for p in parts]),
                                                'preds': {m: np.concatenate([p['preds'][m] for p in parts]) for m in parts[0]['preds']}}
table_pseudo = pd.concat([metric_rows(pseudo, 'pseudo'), metric_rows(pooled, 'pooled')]).sort_index()
display(table_pseudo)

main winners (the same models as in the leaderboard): {'jeonse': 'hgb_detrended_ld_median_recal', 'wolse': 'hgb_detrended_ld_median'}


jeonse / pseudo-test 2023: 12,525 renewal evaluation rows, done in 31 s


jeonse / pseudo-test 2024: 12,439 renewal evaluation rows, done in 35 s


wolse / pseudo-test 2023: 6,228 renewal evaluation rows, done in 14 s


wolse / pseudo-test 2024: 7,558 renewal evaluation rows, done in 18 s


rows  WAPE %  MdAPE %  aggregate bias %  WAPE % (ratio rule also passed)
track  evaluation             model                                                                                                                 
jeonse 2023                   carry-forward                                12525 11.6823   5.0923            3.5018                          11.5141
                              gradient boosting (log ratio, median loss)   12525 14.9362   6.5573           11.8761                          14.7697
                              group median ratio                           12525 14.7378   6.2069           11.4307                          14.5704
                              main winner (hgb_detrended_ld_median_recal)  12525 14.4943  11.5355           -6.5599                          14.4790
       2024                   carry-forward                                12439  7.9816   4.7598           -2.3430                           7.7621
                              gradient boosting (log ratio, median loss)   12439  7.1234   4.2568           -1.6449                           6.9097
                              group median ratio                           12439  8.0574   4.7619            2.4878                           7.8389
                              main winner (hgb_detrended_ld_median_recal)  12439 13.5993  10.2420            1.4429                          13.5924
       pseudo-tests 2023+2024 carry-forward                                24964  9.8166   4.7619            0.5550                           9.6236
                              gradient boosting (log ratio, median loss)   24964 10.9972   5.0000            5.0592                          10.8093
                              group median ratio                           24964 11.3697   5.0000            6.9220                          11.1786
                              main winner (hgb_detrended_ld_median_recal)  24964 14.0431  10.9053           -2.5251                          14.0323
wolse  2023                   carry-forward                                 6228 16.0044   6.9767           -4.4342                          15.5868
                              gradient boosting (log ratio, median loss)    6228 15.4354   5.9787            3.1815                          15.0162
                              group median ratio                            6228 15.3932   5.7707            1.3482                          14.9627
                              main winner (hgb_detrended_ld_median)         6228 41.3674  29.6017          -13.3450                          41.1049
       2024                   carry-forward                                 7558 12.7582   6.0075           -6.1645                          12.1799
                              gradient boosting (log ratio, median loss)    7558 11.4532   5.7235           -0.3005                          10.8704
                              group median ratio                            7558 11.5296   5.3279           -0.6541                          10.9284
                              main winner (hgb_detrended_ld_median)         7558 42.9182  32.4967           -8.5516                          42.7364
       pseudo-tests 2023+2024 carry-forward                                13786 14.2560   6.2500           -5.3662                          13.7524
                              gradient boosting (log ratio, median loss)   13786 13.2906   5.8390            1.3060                          12.7840
                              group median ratio                           13786 13.3122   5.5556            0.2698                          12.7905
                              main winner (hgb_detrended_ld_median)        13786 42.2027  31.0917          -10.7633                          41.9834

**Interpretation.** On the renewal evaluation rows of the pseudo-tests the main winner has a WAPE of 14.0431% (Jeonse) and 42.2027% (Wolse), pooled. The renewal-only models do much better in Wolse (gradient boosting 13.2906%, group median ratio 13.3122%, carry-forward 14.2560%) and the carry-forward is the best in the pooled Jeonse pseudo-tests (9.8166% against 10.9972% for gradient boosting and 11.3697% for the group median ratio). The result depends on the window: in 2023 the carry-forward is best in Jeonse (11.6823% against 14.7378% and 14.9362%, with models fit on the rows of 2022 only) while in 2024 gradient boosting is best (7.1234% against 7.9816% for the carry-forward). In Wolse the models beat the carry-forward in both windows (2023: 15.4354% and 15.3932% against 16.0044%; 2024: 11.4532% and 11.5296% against 12.7582%). The WAPE on rows that also pass the ratio rule is slightly lower for every model.

## 6) 2025, evaluated once

In [8]:
final = {}
for track in config.TRACKS:
    d = data[track]['df']
    test = data[track]['test']
    saved = pd.read_parquet(config.PREDICTIONS_DIR / f'{track}__{chosen[track]}.parquet')['y_pred'].to_numpy()      # 2025 predictions of the main winner (notebook 11)
    assert len(saved) == len(test)
    final[(track, '2025')] = evaluate_window(track, 'final', test, saved)
    print(f"{track} / 2025: {len(final[(track, '2025')]['y']):,} renewal evaluation rows of {int(data[track]['flags'].loc[test.index, 'is_renewal'].sum()):,} renewals and {len(test):,} contracts in TEST")
table_2025 = metric_rows(final, '2025')
display(table_2025)

jeonse / 2025: 15,102 renewal evaluation rows of 15,103 renewals and 37,935 contracts in TEST


wolse / 2025: 10,279 renewal evaluation rows of 12,222 renewals and 57,829 contracts in TEST


rows  WAPE %  MdAPE %  aggregate bias %  WAPE % (ratio rule also passed)
track  evaluation model                                                                                                                 
jeonse 2025       carry-forward                                15102  6.1813   4.7619           -5.6726                           6.0397
                  group median ratio                           15102  3.5753   1.0989           -1.1678                           3.4291
                  gradient boosting (log ratio, median loss)   15102  4.1446   1.7412           -2.2328                           4.0000
                  main winner (hgb_detrended_ld_median_recal)  15102 13.7347  10.4964            4.8818                          13.7177
wolse  2025       carry-forward                                10279 10.5937   5.7239           -7.0494                          10.2803
                  group median ratio                           10279  8.4641   5.1351           -1.9924                           8.1353
                  gradient boosting (log ratio, median loss)   10279  8.4031   4.7235           -1.6339                           8.0833
                  main winner (hgb_detrended_ld_median)        10279 44.3468  35.2583          -18.4175                          44.2387

**Interpretation.** On 2025 (evaluated once, 15,102 of the 15,103 Jeonse renewals and 10,279 of the 12,222 Wolse renewals, the others having no usable previous price) the carry-forward has a WAPE of 6.1813% in Jeonse and 10.5937% in Wolse, with an aggregate bias of -5.6726% and -7.0494% (the new price is higher than the previous one on average). The group median ratio has 3.5753% in Jeonse and 8.4641% in Wolse and gradient boosting 4.1446% and 8.4031%; their bias is -1.1678% and -2.2328% in Jeonse and -1.9924% and -1.6339% in Wolse. In Jeonse the group median ratio is better than gradient boosting; in Wolse both are close. The main winner, which does not know the previous price, has 13.7347% (Jeonse) and 44.3468% (Wolse) on the same rows. The comparison measures the value of knowing the current price, which is information that is not available for new contracts, and it must not be read as a comparison of modeling quality.

In [9]:
rows = []
for (track, name), r in final.items():
    for model, pred in r['preds'].items():
        for right in ['Yes', 'No/Not reported']:
            m = r['right'] == right
            rows.append({'track': track, 'renewal right used': right, 'model': model, 'rows': int(m.sum()), 'WAPE %': wape(r['y'][m], pred[m]),
                         'aggregate bias %': (pred[m].sum() - r['y'][m].sum()) / r['y'][m].sum() * 100})
display(pd.DataFrame(rows).set_index(['track', 'renewal right used', 'model']))

rows  WAPE %  aggregate bias %
track  renewal right used model                                                                      
jeonse Yes                carry-forward                                7164  4.1248           -3.8354
       No/Not reported    carry-forward                                7938  8.6479           -7.8762
       Yes                group median ratio                           7164  1.0081            0.8039
       No/Not reported    group median ratio                           7938  6.6543           -3.5326
       Yes                gradient boosting (log ratio, median loss)   7164  1.8079           -0.4337
       No/Not reported    gradient boosting (log ratio, median loss)   7938  6.9471           -4.3905
       Yes                main winner (hgb_detrended_ld_median_recal)  7164 12.6611            6.2970
       No/Not reported    main winner (hgb_detrended_ld_median_recal)  7938 15.0223            3.1844
wolse  Yes                carry-forward                                3307  7.6863           -6.9830
       No/Not reported    carry-forward                                6972 12.8669           -7.1013
       Yes                group median ratio                           3307  4.3423           -2.2239
       No/Not reported    group median ratio                           6972 11.6868           -1.8115
       Yes                gradient boosting (log ratio, median loss)   3307  4.4246           -2.2756
       No/Not reported    gradient boosting (log ratio, median loss)   6972 11.5138           -1.1322
       Yes                main winner (hgb_detrended_ld_median)        3307 44.0769          -24.1634
       No/Not reported    main winner (hgb_detrended_ld_median)        6972 44.5578          -13.9250

**Interpretation.** The gain comes mostly from renewals with the right used. In Jeonse the WAPE of these renewals falls from 4.1248% (carry-forward) to 1.0081% (group median ratio) and 1.8079% (gradient boosting), against 8.6479%, 6.6543% and 6.9471% for the other renewals. In Wolse the right-used renewals go from 7.6863% to 4.3423% and 4.4246%, and the others from 12.8669% to 11.6868% and 11.5138%. The main winner has 12.6611% and 15.0223% in Jeonse and 44.0769% and 44.5578% in Wolse for the two groups. The renewals without the right used remain much harder to predict than those with it.

## Save the renewal-only runs (scope = renewal_only; not part of the main leaderboard)

In [10]:
IDS = {'carry-forward': 'renewal_carry_forward', 'group median ratio': 'renewal_group_median_ratio',
       'gradient boosting (log ratio, median loss)': 'renewal_hgb_logratio_median'}
for track in config.TRACKS:
    test = data[track]['test']
    f_test = flag_previous(test, track)
    ev = test[f_test['input_ok'].to_numpy()]
    r = final[(track, '2025')]
    for label, model_id in IDS.items():
        meta = {'scope': SCOPE, 'product': 'what will a tenant pay on a future renewal, given the current price',
                'uses': 'previous_deposit_10k_krw' if track == 'jeonse' else 'previous_monthly_rent_10k_krw', 'not_for': 'new contracts and the market projection',
                'evaluation_rows': 'renewals with a positive previous price not above the hard cap, TEST 2025', 'hard_cap': PRICE_CAP[track], 'ratio_bounds': list(RATIO_BOUNDS),
                'training_rows': 'renewals passing all plausibility rules, fit window before 2025-01-01',
                'params': tuned[(track, 'final')] if 'boosting' in label else None,
                'pseudo_test_wape': {w['name']: float(all_metrics(pseudo[(track, w['name'])]['y'], pseudo[(track, w['name'])]['preds'][label])['wape_pct']) for w in WINDOWS}}
        save_run(track, model_id, r['y'], r['preds'][label], meta=meta, context=ev[['district_name', 'contract_date']])
print('saved', len(IDS) * len(config.TRACKS), 'runs with scope =', SCOPE, ':', list(IDS.values()))
from rent_model.compare import load_runs
assert not load_runs()['model_id'].str.startswith('renewal_').any()
print('assertion passed: the main leaderboard loader does not list the renewal-only runs')

saved 6 runs with scope = renewal_only : ['renewal_carry_forward', 'renewal_group_median_ratio', 'renewal_hgb_logratio_median']


assertion passed: the main leaderboard loader does not list the renewal-only runs


**Interpretation.** Six runs are saved (three models for each track) with `scope = renewal_only` in their meta: the prediction of each model on the 2025 renewal evaluation rows. The assertion confirms that the loader of the main leaderboard does not list them, so they are neither ranked against the market models nor used by the market projection.

## Summary

## Conclusion

- **Scope.** A separate product for renewals, with the previous price as a domain assumption; it does not apply to new contracts (most of the TEST rows are not renewals) and it is not part of the market projection.
- **Data.** Renewals are 39.8128% (Jeonse) and 21.1347% (Wolse) of the TEST rows; 569 and 5,582 renewals are flagged by the plausibility rule (in Wolse mostly because the previous contract had no monthly rent).
- **Policy.** In Jeonse, 0.9859 of the renewals with the right used are at or below a ratio of 1.05, consistent with a 5% limit; the data do not show it for Wolse.
- **Accuracy on 2025.** Carry-forward 6.1813% (Jeonse) and 10.5937% (Wolse); group median ratio 3.5753% and 8.4641%; gradient boosting 4.1446% and 8.4031%; main winner on the same rows 13.7347% and 44.3468%. The gain of the models over the carry-forward is clear in 2025 but not in every pseudo-test window (Jeonse 2023).
- **Limits.** The model is only valid for renewals with a known previous price; the previous contract has no date in the data; 2025 was inspected in earlier iterations of the project.